# Coupled Dynamics: Distance vs Private Information
Notebook to visualize the coupled dynamics over different timescales.


In [ ]:
import os
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from sklearn.cluster import KMeans
from scipy.special import softmax
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
wandb_dir = os.environ.get("WANDB_DIR", "./")
data_path = os.path.join(wandb_dir, "data", "v9arred3_jmrbbem4", "merged_agent_stats.parquet")
dist_data_path = os.path.join(wandb_dir, "data", "v9arred3_jmrbbem4", "merged_group_distances.parquet")

print("Loading data...")
df = pd.read_parquet(data_path)
df_dist = pd.read_parquet(dist_data_path)

if 'group_id' in df.columns:
    df['group_id'] = df['group_id'].astype(float).astype(int)
if 'group_id' in df_dist.columns:
    df_dist['group_id'] = df_dist['group_id'].astype(float).astype(int)
    
df_dist['unique_group'] = df_dist['run_id'].astype(str) + '_' + df_dist['group_id'].astype(str)

print("Calculating probabilities...")
genes_cols = ['gene_priv', 'gene_soc', 'gene_none']
probs = softmax(df[genes_cols].values, axis=1)
df['prob_priv'] = probs[:, 0]
df['prob_soc'] = probs[:, 1]
df['prob_none'] = probs[:, 2]

print("Calculating performance clusters...")
group_perf = df.groupby(['run_id', 'group_id', 'step', 'close_target_index'])['current_fitness'].median().reset_index()
group_perf = group_perf[group_perf['step'] < 199000]
group_perf['unique_group'] = group_perf['run_id'].astype(str) + '_' + group_perf['group_id'].astype(str)

n_clusters = 6
conditions = sorted(group_perf['close_target_index'].unique())
cluster_groups_by_cond = {}

for i, cond in enumerate(conditions):
    cond_data = group_perf[group_perf['close_target_index'] == cond].copy()
    heatmap_data = cond_data.pivot(index='unique_group', columns='step', values='current_fitness')
    X_cluster = heatmap_data.fillna(0).values
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit(X_cluster)
    cluster_labels = kmeans.labels_

    group_mean_perf = heatmap_data.mean(axis=1).values
    cluster_avg_perf = {c: group_mean_perf[cluster_labels == c].mean() for c in range(n_clusters)}
    sorted_clusters = sorted(cluster_avg_perf.keys(), key=lambda c: cluster_avg_perf[c])

    cluster_groups = []
    for c in sorted_clusters:
        idx_in_cluster = np.where(cluster_labels == c)[0]
        sorted_idx_in_cluster = idx_in_cluster[np.argsort(group_mean_perf[idx_in_cluster])]
        groups = heatmap_data.index[sorted_idx_in_cluster].tolist()
        cluster_groups.append(groups)
    
    cluster_groups_by_cond[cond] = cluster_groups

print("Aggregating gene probabilities and merging with distance...")
group_cols = ['run_id', 'group_id', 'step']
df_genes = df.groupby(group_cols)[['prob_priv', 'prob_soc', 'prob_none']].mean().reset_index()

# Properly merge by generation index
# df step is 999, 1999, etc. So step//1000 + 1 = generation 1, 2...
df_genes['gen'] = (df_genes['step'] // 1000) + 1
df_dist['gen'] = (df_dist['tick'] // 1000) + 1

df_merged = pd.merge(df_dist, df_genes[['run_id', 'group_id', 'gen', 'prob_priv', 'prob_soc', 'prob_none']], on=['run_id', 'group_id', 'gen'], how='inner')
print("Data preparation complete.")


## 1. Phase Portrait: Distance vs. Private Info
Visualizing trajectories mapped across the state space of distance and genetic reliance.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
cond_titles = {0: "Init on Good Target", 1: "Init on Bad Target"}
time_periods = [(0, 100000), (100000, 200000)]
time_titles = ["0 - 100k Ticks", "100k - 200k Ticks"]

os.makedirs('figures', exist_ok=True)

lc_cols = {0: None, 1: None}

for row, cond_val in enumerate([0, 1]):
    cluster_groups = cluster_groups_by_cond.get(cond_val, [])
    cond_data = df_merged[df_merged['close_target_index'] == cond_val]
    
    for col, (t_start, t_end) in enumerate(time_periods):
        ax = axes[row, col]
        phase_data = cond_data[(cond_data['tick'] >= t_start) & (cond_data['tick'] <= t_end)]
        
        if len(phase_data) == 0:
            continue
            
        for c, groups in enumerate(cluster_groups):
            cluster_df = phase_data[phase_data['unique_group'].isin(groups)]
            if len(cluster_df) == 0: continue
            
            # Aggregate by tick to get mean behavior at each tick across all groups in the cluster
            agg_df = cluster_df.groupby('tick').agg({'mean_good_dist': 'mean', 'prob_priv': 'mean'}).reset_index()
            
            # Smooth over half a generation (500 ticks) to remove intra-generation high-frequency noise
            x = agg_df['mean_good_dist'].rolling(window=500, min_periods=1).mean().values
            y = agg_df['prob_priv'].rolling(window=500, min_periods=1).mean().values
            ticks = agg_df['tick'].values
            
            # Subsample for rendering performance and clarity (plot 1 point every 100 ticks)
            x = x[::1000]
            y = y[::1000]
            ticks = ticks[::1000]
            
            if len(x) < 2:
                continue
                
            points = np.array([x, y]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            
            # Using linear color scale for time
            norm = mcolors.Normalize(vmin=t_start, vmax=t_end)
            
            lc = LineCollection(segments, cmap='viridis', norm=norm, linewidth=3, alpha=0.9)
            lc.set_array(ticks[:-1])
            ax.add_collection(lc)
            lc_cols[col] = lc
            
            # Start and End markers
            ax.scatter(x[0], y[0], marker='o', color='black', s=50, edgecolor='white', zorder=3)
            ax.scatter(x[-1], y[-1], marker='X', color='black', s=100, edgecolor='white', zorder=3)
            ax.text(x[-1], y[-1], f" C{6-c}", fontsize=12, fontweight='bold', color='black', verticalalignment='bottom', zorder=4)

        ax.set_xlim(0, 10)
        ax.set_ylim(0, 0.4)
        if row == 1:
            ax.set_xlabel("Mean Distance to Good Target", fontsize=16)
        if col == 0:
            ax.set_ylabel("Probability of Private Info", fontsize=16)
            
        ax.set_title(f"{cond_titles[cond_val]} | {time_titles[col]}", fontsize=18, pad=10)
        ax.grid(True, linestyle='--', alpha=0.5)

cbar0 = fig.colorbar(lc_cols[0], ax=axes[:, 0].tolist(), fraction=0.046, pad=0.04)
cbar0.set_label("Tick Time 0 - 100k", fontsize=14)

cbar1 = fig.colorbar(lc_cols[1], ax=axes[:, 1].tolist(), fraction=0.046, pad=0.04)
cbar1.set_label("Tick Time 100k - 200k", fontsize=14)
plt.savefig('figures/phase_portrait_distance_vs_priv_2x2.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/phase_portrait_distance_vs_priv_2x2.pdf', dpi=300, bbox_inches='tight')
plt.show()


## 2. Dual-Axis Time Series
Plotting Distance (Left Y-Axis, solid lines) and Private Info (Right Y-Axis, dashed lines) across time to identify lead-lag dynamics.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
cond_titles = {0: "Init on Good Target", 1: "Init on Bad Target"}
time_periods = [(0, 100000), (100000, 200000)]
time_titles = ["0 - 100k Ticks", "100k - 200k Ticks"]

# Standard qualitative colors for 6 clusters
colors = plt.cm.tab10.colors[:6]

for row, cond_val in enumerate([0, 1]):
    cluster_groups = cluster_groups_by_cond.get(cond_val, [])
    cond_data = df_merged[df_merged['close_target_index'] == cond_val]
    
    for col, (t_start, t_end) in enumerate(time_periods):
        ax1 = axes[row, col]
        ax2 = ax1.twinx()
        
        phase_data = cond_data[(cond_data['tick'] >= t_start) & (cond_data['tick'] <= t_end)]
        
        if len(phase_data) == 0:
            continue
            
        for c, groups in enumerate(cluster_groups):
            cluster_df = phase_data[phase_data['unique_group'].isin(groups)]
            if len(cluster_df) == 0: continue
            
            # Aggregate by tick
            agg_df = cluster_df.groupby('tick').agg({'mean_good_dist': 'mean', 'prob_priv': 'mean'}).reset_index()
            
            # Smooth
            x_time = agg_df['tick'].values
            y_dist = agg_df['mean_good_dist'].rolling(window=500, min_periods=1).mean().values
            y_priv = agg_df['prob_priv'].rolling(window=500, min_periods=1).mean().values
            
            # Subsample
            x_time = x_time[::100]
            y_dist = y_dist[::100]
            y_priv = y_priv[::100]
            
            if len(x_time) < 2: continue
            
            # Distance on left axis (solid line)
            ax1.plot(x_time, y_dist, color=colors[c], linestyle='-', linewidth=2.5, label=f"C{6-c} Dist", alpha=0.9)
            
            # Prob Priv on right axis (dashed line)
            ax2.plot(x_time, y_priv, color=colors[c], linestyle='--', linewidth=2.5, label=f"C{6-c} Priv", alpha=0.9)
            
        ax1.set_ylim(0, 12)
        ax2.set_ylim(0, 0.4)
        
        if row == 1:
            ax1.set_xlabel("Tick", fontsize=16)
        if col == 0:
            ax1.set_ylabel("Mean Distance to Good Target", fontsize=16, color='black')
        if col == 1:
            ax2.set_ylabel("Probability of Private Info", fontsize=16, color='black')
        else:
            ax2.set_yticklabels([]) # Hide right labels on the left column to avoid clutter
            
        ax1.set_title(f"{cond_titles[cond_val]} | {time_titles[col]}", fontsize=18, pad=10)
        ax1.grid(True, linestyle=':', alpha=0.5)
        ax1.set_xscale("log", nonpositive='mask')
        ax2.set_xscale("log", nonpositive='mask')

import matplotlib.lines as mlines
legend_elements = [
    mlines.Line2D([0], [0], color='k', linestyle='-', linewidth=2.5, label='Distance to Good (Left Axis)'),
    mlines.Line2D([0], [0], color='k', linestyle='--', linewidth=2.5, label='Prob Private (Right Axis)')
]
for c in range(6):
    legend_elements.append(mlines.Line2D([0], [0], color=colors[c], linestyle='-', linewidth=4, label=f'Cluster {6-c}'))

fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4, fontsize=14)
plt.tight_layout()
plt.subplots_adjust(top=0.9) # Make room for legend
plt.savefig('figures/dual_axis_distance_vs_priv_2x2.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/dual_axis_distance_vs_priv_2x2.pdf', dpi=300, bbox_inches='tight')
plt.show()


## 4. Small Multiples: Distance & Private Info Relationship

In [ ]:
cond_titles = {0: "Init on Good Target", 1: "Init on Bad Target"}

# We will focus on the first 30 generations of each phase
# Note: x-axis will be in Generations
time_windows = [
    (0, 30000),      # Phase 1: 0 - 30k ticks
    (100000, 130000) # Phase 2: 100k - 130k ticks
]
window_titles = ["Phase 1 (Gens 0-30)", "Phase 2 (Gens 100-130)"]

dist_color = 'tab:blue'
priv_color = 'tab:orange'

for cond_val in [0, 1]:
    # Create a separate figure for each condition
    fig, axes = plt.subplots(2, 6, figsize=(24, 8), sharey='row')
    fig.suptitle(f"{cond_titles[cond_val]}", fontsize=22, fontweight='bold', y=1.05)
    
    cluster_groups = cluster_groups_by_cond.get(cond_val, [])
    cond_data = df_merged[df_merged['close_target_index'] == cond_val]
    
    for row, (t_start, t_end) in enumerate(time_windows):
        phase_data = cond_data[(cond_data['tick'] >= t_start) & (cond_data['tick'] <= t_end)]
        
        for col in range(6):
            # Reverse order so C1 is on the left (col=0) and C6 is on the right (col=5)
            # Original list has C6 at index 0, so C1 is at index 5.
            c_idx = 5 - col
            
            ax1 = axes[row, col]
            ax2 = ax1.twinx()
            
            if c_idx >= len(cluster_groups) or c_idx < 0:
                ax1.set_visible(False)
                ax2.set_visible(False)
                continue
                
            groups = cluster_groups[c_idx]
            cluster_df = phase_data[phase_data['unique_group'].isin(groups)]
            if len(cluster_df) == 0: 
                continue
                
            # Aggregate by tick
            agg_df = cluster_df.groupby('tick').agg({'mean_good_dist': 'mean', 'prob_priv': 'mean'}).reset_index()
            
            # Smooth (smoothing over 100 ticks since the window is small)
            x_time = agg_df['tick'].values
            y_dist = agg_df['mean_good_dist'].rolling(window=100, min_periods=1).mean().values
            y_priv = agg_df['prob_priv'].rolling(window=100, min_periods=1).mean().values
            
            # Convert ticks to generations
            x_gen = x_time / 1000.0
            
            # Subsample for rendering speed
            x_gen = x_gen[::50]
            y_dist = y_dist[::50]
            y_priv = y_priv[::50]
            
            if len(x_gen) < 2: continue
            
            # Plot
            l1, = ax1.plot(x_gen, y_dist, color=dist_color, linestyle='-', linewidth=2.5, alpha=0.9, label="Distance")
            l2, = ax2.plot(x_gen, y_priv, color=priv_color, linestyle='--', linewidth=2.5, alpha=0.9, label="Prob Priv")
            
            # Set axis limits
            ax1.set_ylim(0, 12)
            ax2.set_ylim(0, 0.5)
            # Make sure x-limits strictly match the requested window
            ax1.set_xlim(t_start / 1000.0, t_end / 1000.0)
            
            # Formatting
            if row == 0:
                ax1.set_title(f"Cluster {col+1}", fontsize=16, fontweight='bold')
            
            if row == 1:
                ax1.set_xlabel("Generation", fontsize=14)
                
            if col == 0:
                ax1.set_ylabel(f"{window_titles[row]}\n\nDistance to Good", fontsize=14, color=dist_color)
                ax1.tick_params(axis='y', labelcolor=dist_color)
            else:
                ax1.tick_params(labelleft=False)
                
            if col == 5:
                ax2.set_ylabel("Prob of Private Info", fontsize=14, color=priv_color, rotation=-90, labelpad=20)
                ax2.tick_params(axis='y', labelcolor=priv_color)
            else:
                ax2.tick_params(labelright=False)
                
            ax1.grid(True, linestyle=':', alpha=0.5)

    # Master legend for this figure
    fig.legend(handles=[l1, l2], loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.subplots_adjust(wspace=0.1, hspace=0.2)
    plt.savefig(f'figures/small_multiples_cond_{cond_val}.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Flow Diagram (Streamplot) for All Groups

In [ ]:
import numpy.ma as ma
from scipy.stats import binned_statistic_2d
from scipy.ndimage import gaussian_filter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 1. Prepare generation-level flow data for ALL groups using vectorized pandas
gen_df_all = df_merged.groupby(['unique_group', 'gen']).agg({
    'mean_good_dist': 'mean', 
    'prob_priv': 'mean', 
    'close_target_index': 'first'
}).reset_index()

gen_df_all = gen_df_all.sort_values(['unique_group', 'gen'])

# Calculate differentials between generations in a fully vectorized way
gen_df_all['dx'] = gen_df_all.groupby('unique_group')['mean_good_dist'].diff()
gen_df_all['dy'] = gen_df_all.groupby('unique_group')['prob_priv'].diff()

flow_df = gen_df_all.dropna(subset=['dx', 'dy'])

# 2. Setup grid and windows
grid_res = 50
x_bins = np.linspace(0, 6, grid_res)
y_bins = np.linspace(0, 0.5, grid_res)
X, Y = np.meshgrid(x_bins[:-1] + np.diff(x_bins)/2, y_bins[:-1] + np.diff(y_bins)/2)

cond_titles = {0: "Init on Good Target", 1: "Init on Bad Target"}

# 1x2 grid for Phase 1 (Gens 0-100) only
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for col, cond_val in enumerate([0, 1]):
    cond_flow = flow_df[flow_df['close_target_index'] == cond_val]
    current_cmap = 'Blues' if cond_val == 0 else 'Reds'
    
    ax = axes[col]
    
    # Filter for full 100 generations of Phase 1
    phase_data = cond_flow[(cond_flow['gen'] >= 1) & (cond_flow['gen'] <= 5)]
    if len(phase_data) == 0:
        continue
        
    U, _, _, _ = binned_statistic_2d(phase_data['mean_good_dist'], phase_data['prob_priv'], phase_data['dx'], statistic='mean', bins=[x_bins, y_bins])
    V, _, _, _ = binned_statistic_2d(phase_data['mean_good_dist'], phase_data['prob_priv'], phase_data['dy'], statistic='mean', bins=[x_bins, y_bins])
    Counts, _, _, _ = binned_statistic_2d(phase_data['mean_good_dist'], phase_data['prob_priv'], phase_data['dx'], statistic='count', bins=[x_bins, y_bins])
    
    U, V, Counts = U.T, V.T, Counts.T
    
    # Smooth velocity fields
    U_smooth = gaussian_filter(np.nan_to_num(U, 0), sigma=1.5)
    V_smooth = gaussian_filter(np.nan_to_num(V, 0), sigma=1.5)
    
    # Mask where density is low so we don't draw arrows in empty space
    arrow_density = gaussian_filter(Counts, sigma=1.5)
    threshold = arrow_density.max() * 0.15
    mask = arrow_density < threshold
    U_masked = ma.array(U_smooth, mask=mask)
    V_masked = ma.array(V_smooth, mask=mask)
    
    # Heatmap of end states for this period
    end_data = phase_data.groupby(['unique_group']).last().reset_index()
    Counts_end, _, _, _ = binned_statistic_2d(end_data['mean_good_dist'], end_data['prob_priv'], end_data['mean_good_dist'], statistic='count', bins=[x_bins, y_bins])
    Counts_end = Counts_end.T
    density_end = gaussian_filter(Counts_end, sigma=1.5)
    
    heatmap_data = ma.array(density_end, mask=(density_end < 1e-4))
    extent = [x_bins.min(), x_bins.max(), y_bins.min(), y_bins.max()]
    
    # Plot the background density heatmap
    mesh = ax.imshow(heatmap_data, extent=extent, origin='lower', cmap=current_cmap, alpha=0.6, interpolation='bilinear', zorder=1, aspect='auto')
    
    # Plot the smooth streamplot
    try:
        stream = ax.streamplot(X, Y, U_masked, V_masked, color='black', linewidth=1.5, density=1.5, arrowsize=1.5, zorder=2)
    except ValueError:
        pass
        
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 0.5)
    
    ax.set_xlabel("Mean Distance to Good Target", fontsize=16)
    if col == 0:
        ax.set_ylabel("Probability of Private Info", fontsize=16)
        
    ax.set_title(f"{cond_titles[cond_val]} | Phase 1 (Gens 0-5)", fontsize=18, pad=10)
    ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('figures/streamplot_distance_vs_priv_phase1.png', dpi=300, bbox_inches='tight')
plt.show()